In [1]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
GPU available: True
GPU name: Tesla T4


In [2]:
import os, random, numpy as np

def set_seed(seed: int = 42):
    """Make results repeatable across runs and machines."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True   # reproducible
    torch.backends.cudnn.benchmark = False       # slightly slower, but stable
    print(f"[repro] all seeds set to {seed}")

Tensor Board Login - Fake numbers appear for initial

In [3]:
from torch.utils.tensorboard import SummaryWriter

# This standalone demo defines its own numbers, so it runs with no errors.
writer = SummaryWriter(log_dir="results/runs/demo")
for epoch in range(5):                      # pretend we trained 5 epochs
    train_loss = 1.0 / (epoch + 1)          # fake numbers, just for the demo
    val_loss   = 1.2 / (epoch + 1)
    val_acc    = 0.6 + 0.07 * epoch
    writer.add_scalar("loss/train", train_loss, epoch)
    writer.add_scalar("loss/val",   val_loss,   epoch)
    writer.add_scalar("acc/val",    val_acc,    epoch)
writer.close()
print("Logged 5 fake epochs to results/runs/demo — open TensorBoard to see the curves.")

Logged 5 fake epochs to results/runs/demo — open TensorBoard to see the curves.


In [4]:
%reload_ext tensorboard

# Baseline Establishment

In [5]:
!pip install timm --quiet

import os, json, copy, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
from tqdm import tqdm
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, accuracy_score)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
assert torch.cuda.is_available(), "GPU is OFF — turn on the accelerator (step 2 above)."

Device: cuda


# Reproducibility

In [6]:
def set_seed(seed: int = 42):
    """Set the random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Data

In [7]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [8]:
def build_transforms(img_size: int = 224):
  train_tf = transforms.Compose(
    [
      transforms.Resize((img_size, img_size)),
      transforms.RandomCrop(img_size, padding=8, padding_mode='reflect'),
      transforms.RandomHorizontalFlip(p=0.5),
      transforms.RandomRotation(degrees=10),
      transforms.ColorJitter(brightness=0.2, contrast=0.2),
      transforms.Grayscale(num_output_channels=3),
      transforms.ToTensor(),
      transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ]
  )

  eval_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
  ])
  return train_tf, eval_tf

In [9]:
class TransformSubset(Dataset):
  """Wraps a Subset of an ImageFolder so train/val/test can each use a different transform."""

  def __init__(self, subset: Subset, transform):
    self.subset = subset
    self.transform = transform

  def __len__(self):
    return len(self.subset)

  def __getitem__(self, idx):
    img, label = self.subset[idx]
    if self.transform is not None:
      img = self.transform(img)
    return img, label

In [10]:
def stratified_split(dataset: ImageFolder, seed: int = 42):
  """70/15/15 stratified split of dataset into train, val, and test subsets."""

  targets = np.array(dataset.targets)
  indices = np.arange(len(dataset))

  train_idx, temp_idx = train_test_split(
    indices, test_size=0.3, stratify=targets, random_state=seed
  )

  val_idx, test_idx = train_test_split(
    temp_idx,
    test_size= 0.50,
    stratify=targets[temp_idx],
    random_state = seed,
  )

  return train_idx, val_idx, test_idx

In [11]:
def build_dataloaders(data_dir: str, img_size: int, batch_size: int, seed: int, num_workers: int = 4):
  # ImageFolder expects: data_dir/<class_name>/*.png
  # Load once without transform so PIL images can be transformed differently per split.
  def only_images_folder(path):
    p = Path(path)
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".gif"}
    return p.parent.name.lower() == "images" and p.suffix.lower() in valid_exts
  base_dataset = ImageFolder(root=data_dir, is_valid_file=only_images_folder)

  train_idx, val_idx, test_idx = stratified_split(base_dataset, seed=seed)
  train_tf, eval_tf = build_transforms(img_size)

  train_ds = TransformSubset(Subset(base_dataset, train_idx), train_tf)
  val_ds = TransformSubset(Subset(base_dataset, val_idx), eval_tf)
  test_ds = TransformSubset(Subset(base_dataset, test_idx), eval_tf)

  pin_memory = torch.cuda.is_available()
  train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
  val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
  test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

  class_names = base_dataset.classes
  train_targets = np.array(base_dataset.targets)[train_idx]
  datasets = {"train": train_ds, "val": val_ds, "test": test_ds}
  return train_loader, val_loader, test_loader, class_names, train_targets, datasets

In [12]:
def compute_class_weights(train_targets: np.ndarray, num_classes: int) -> torch.Tensor:
  counts = np.bincount(train_targets, minlength=num_classes).astype(np.float32)
  counts[counts == 0] = 1.0  # avoid div-by-zero
  weights = counts.sum() / (num_classes * counts)
  return torch.tensor(weights, dtype=torch.float32)

# Training Evaluation / Loops

In [13]:
def run_epoch(model, loader, criterion, optimizer, device, train, desc=""):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in tqdm(loader, desc=desc, leave=False):
            images, labels = images.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total   += images.size(0)
    return total_loss / total, correct / total

In [14]:
def train_phase(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, patience, phase_name, output_dir):
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    no_improve = 0
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, device, True,  f"{phase_name} {epoch}/{epochs} train")
        va_loss, va_acc = run_epoch(model, val_loader,   criterion, optimizer, device, False, f"{phase_name} {epoch}/{epochs} val")
        if scheduler is not None: scheduler.step()
        print(f"[{phase_name}] epoch {epoch}/{epochs} "
              f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} val_loss={va_loss:.4f} val_acc={va_acc:.4f}")
        if va_loss < best_val_loss:
            best_val_loss = va_loss; best_state = copy.deepcopy(model.state_dict()); no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"[{phase_name}] early stopping at epoch {epoch}"); break
    model.load_state_dict(best_state)
    return model

In [15]:
@torch.no_grad()
def evaluate(model, loader, class_names, device, output_dir):
    model.eval(); ys, ps, probs = [], [], []
    for images, labels in loader:
        out = model(images.to(device))
        p = torch.softmax(out, 1).cpu().numpy()
        ys.extend(labels.numpy()); ps.extend(p.argmax(1)); probs.extend(p)
    ys, ps, probs = np.array(ys), np.array(ps), np.array(probs)

    print("Test accuracy:", round(accuracy_score(ys, ps), 4))
    print(classification_report(ys, ps, target_names=class_names, digits=4))
    cm = confusion_matrix(ys, ps); print("Confusion matrix:\n", cm)
    try:
        auc = roc_auc_score(ys, probs, multi_class="ovr", average="macro")
        print("Macro ROC-AUC:", round(auc, 4))
    except ValueError:
        auc = None

    rep = classification_report(ys, ps, target_names=class_names, digits=4, output_dict=True)
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(Path(output_dir)/"confusion_matrix.csv")
    pd.DataFrame([{
        "accuracy": accuracy_score(ys, ps),
        "macro_f1": rep["macro avg"]["f1-score"],
        "macro_recall": rep["macro avg"]["recall"],
        "macro_precision": rep["macro avg"]["precision"],
        "roc_auc_macro": auc,
    }]).to_csv(Path(output_dir)/"summary_metrics.csv", index=False)
    return rep

### Build the Loaders

In [16]:
# On Kaggle, after adding the "COVID-19 Radiography Database" input, the path is:
DATA_DIR = "/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset"
# (If a path error appears, run  !ls /kaggle/input  and adjust to what you see.)

train_loader, val_loader, test_loader, class_names, train_targets, datasets = build_dataloaders(
    data_dir=DATA_DIR, img_size=224, batch_size=32, seed=42, num_workers=4
)
num_classes = len(class_names)
print("Classes:", class_names)   # ['COVID','Lung_Opacity','Normal','Viral Pneumonia']

# Same class-weighted loss Member 2 used (handles the class imbalance fairly):
class_weights = compute_class_weights(train_targets, num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

Classes: ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']


# Build ResNet50

In [17]:
def build_resnet50(num_classes=4):
    return timm.create_model("resnet50", pretrained=True, num_classes=num_classes)

In [18]:
def unfreeze_final_blocks_resnet(model, num_blocks=1):
    """Phase 2: unfreeze the head + the last conv block(s) for fine-tuning."""
    for p in model.parameters():
        p.requires_grad = False
    keep = ["fc"] + [f"layer{4 - i}" for i in range(num_blocks)]
    for name, p in model.named_parameters():
        if any(name.startswith(k) for k in keep):
            p.requires_grad = True

In [19]:
def freeze_backbone(model):
    """Phase 1: freeze everything except the classifier head (named 'fc' in ResNet)."""
    for name, p in model.named_parameters():
        p.requires_grad = ("fc" in name or "classifier" in name)

In [20]:
def build_optimizer(model, name, lr, weight_decay):
    params = [p for p in model.parameters() if p.requires_grad]
    if name == "adamw": return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if name == "adam":  return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd":   return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(name)

In [21]:
def build_scheduler(optimizer, name, epochs):
    if name == "cosine": return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    if name == "none":   return None
    raise ValueError(name)

## Train the baseline ResNet50 Model

In [22]:
OUT = "./runs/resnet50_baseline"; os.makedirs(OUT, exist_ok=True)
set_seed(42)
model = build_resnet50(num_classes).to(device)

# Phase 1: frozen backbone, train the head only
freeze_backbone(model)
opt1   = build_optimizer(model, "adamw", lr=1e-3, weight_decay=1e-4)
sched1 = build_scheduler(opt1, "cosine", epochs=15)
model  = train_phase(model, train_loader, val_loader, criterion, opt1, sched1,
                     device, epochs=15, patience=5, phase_name="phase1_frozen", output_dir=OUT)

# Phase 2: unfreeze the last block, fine-tune with a tiny learning rate
unfreeze_final_blocks_resnet(model, num_blocks=1)
opt2   = build_optimizer(model, "adamw", lr=1e-5, weight_decay=1e-4)
sched2 = build_scheduler(opt2, "cosine", epochs=40)
model  = train_phase(model, train_loader, val_loader, criterion, opt2, sched2,
                     device, epochs=40, patience=5, phase_name="phase2_finetune", output_dir=OUT)

# Save the trained model (same checkpoint format as Member 2)
torch.save({"model_state_dict": model.state_dict(), "class_names": class_names},
           Path(OUT) / "resnet50_baseline.pt")
print("Saved:", Path(OUT) / "resnet50_baseline.pt")

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

[phase1_frozen] epoch 1/15 train_loss=0.9192 train_acc=0.6435 val_loss=0.7240 val_acc=0.7609


[phase1_frozen] epoch 2/15 train_loss=0.6909 train_acc=0.7222 val_loss=0.6222 val_acc=0.7757


[phase1_frozen] epoch 3/15 train_loss=0.6231 train_acc=0.7446 val_loss=0.5852 val_acc=0.7877


[phase1_frozen] epoch 4/15 train_loss=0.5854 train_acc=0.7552 val_loss=0.5387 val_acc=0.7962


...

[phase2_finetune] epoch 36/40 train_loss=0.3643 train_acc=0.8427 val_loss=0.3491 val_acc=0.8513


[phase2_finetune] epoch 37/40 train_loss=0.3591 train_acc=0.8421 val_loss=0.3500 val_acc=0.8526


[phase2_finetune] epoch 38/40 train_loss=0.3626 train_acc=0.8411 val_loss=0.3591 val_acc=0.8498


[phase2_finetune] epoch 39/40 train_loss=0.3656 train_acc=0.8387 val_loss=0.3563 val_acc=0.8551


[phase2_finetune] epoch 40/40 train_loss=0.3567 train_acc=0.8439 val_loss=0.3560 val_acc=0.8523
Saved: runs/resnet50_baseline/resnet50_baseline.pt


In [23]:
results = evaluate(model, test_loader, class_names, device, OUT)

Test accuracy: 0.857
                 precision    recall  f1-score   support

          COVID     0.7610    0.8579    0.8066       542
   Lung_Opacity     0.8642    0.8182    0.8405       902
         Normal     0.8932    0.8698    0.8814      1529
Viral Pneumonia     0.8507    0.9307    0.8889       202

       accuracy                         0.8570      3175
      macro avg     0.8423    0.8692    0.8544      3175
   weighted avg     0.8597    0.8570    0.8575      3175

Confusion matrix:
 [[ 465   24   49    4]
 [  64  738   99    1]
 [  81   90 1330   28]
 [   1    2   11  188]]
Macro ROC-AUC: 0.9678
